# Scenario Runner

Live scenario reset, parent-order execution, venue fill breakdown, and run controls for the SOR engine API.

Edit these values before running the next cell. `quantity`, `max_route_attempts`, and `route_timeout_millis` control how the engine reslices until the order is filled or the timeout/attempt limit is reached.

In [1]:
from IPython.display import HTML, display
import importlib
import pandas as pd
import adaptive_quantum_sor.client as sor_client_module

sor_client_module = importlib.reload(sor_client_module)
SorNotebookClient = sor_client_module.SorNotebookClient

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 0)

def display_wrapped(frame):
    if frame is None or frame.empty:
        display(frame)
        return
    html_table = frame.to_html(index=False, escape=True)
    display(HTML("""
    <style>
    table.sor-wrap-table {border-collapse:collapse; width:100%; table-layout:auto; margin:8px 0 18px 0;}
    table.sor-wrap-table th, table.sor-wrap-table td {
        border:1px solid #d6dae1; padding:6px 8px; vertical-align:top;
        white-space:normal; overflow-wrap:anywhere; word-break:break-word;
        max-width:520px; text-align:left;
    }
    table.sor-wrap-table th {background:#f8fafc; color:#111827;}
    </style>
    """ + html_table.replace('class="dataframe"', 'class="sor-wrap-table"')))

client = SorNotebookClient()

scenario_id = 'notebook-scenario'
seed = 42
ticks = 10
reset_mode = 'PURGE_AND_REPOPULATE'

instrument_id = 0
side = 1  # 1=BUY, -1=SELL
quantity = 2500
urgency_id = 0
max_route_attempts = 8
route_timeout_millis = 1000

parent_order = {
    'instrumentId': instrument_id,
    'side': side,
    'quantity': quantity,
    'urgencyId': urgency_id,
}

display_wrapped(pd.DataFrame([{
    'scenarioId': scenario_id,
    'seed': seed,
    'ticks': ticks,
    'resetMode': reset_mode,
    'maxRouteAttempts': max_route_attempts,
    'routeTimeoutMillis': route_timeout_millis,
    **parent_order,
}]))


scenarioId,seed,ticks,resetMode,maxRouteAttempts,routeTimeoutMillis,instrumentId,side,quantity,urgencyId
notebook-scenario,42,10,PURGE_AND_REPOPULATE,8,1000,0,1,2500,0


In [2]:
reset = client.reset_scenario(scenario_id, seed=seed, ticks=ticks, reset_mode=reset_mode)
run = client.run_scenario(
    scenario_id,
    seed=seed,
    ticks=ticks,
    reset_mode=reset_mode,
    parent_orders=[parent_order],
    max_route_attempts=max_route_attempts,
    route_timeout_millis=route_timeout_millis,
)
summary = client.scenario_summary_dataframe()
events = client.scenario_events_dataframe()

parent_order_results = pd.DataFrame([
    {key: value for key, value in result.items() if key != 'fills'}
    for result in run.get('parentOrderResults', [])
])
fill_rows = []
for result in run.get('parentOrderResults', []):
    for fill in result.get('fills', []):
        fill_rows.append(fill)
fills = pd.DataFrame(fill_rows)

display_wrapped(pd.DataFrame([reset]).add_prefix('reset.'))
display_wrapped(pd.DataFrame([run]).drop(columns=['parentOrderResults'], errors='ignore').add_prefix('run.'))
display_wrapped(parent_order_results.add_prefix('order.'))
display_wrapped(fills.add_prefix('fill.'))
display_wrapped(summary.add_prefix('summary.'))
event_columns = ['sequence', 'eventId', 'timestampNanos', 'event', 'componentId', 'eventType', 'correlationId', 'message']
display_wrapped(events[[column for column in event_columns if column in events.columns]])


reset.scenarioId,reset.resetMode,reset.replaySafe,reset.clearedState,reset.keptState,reset.repopulatedState
notebook-scenario,PURGE_AND_REPOPULATE,True,orders|stats|outcomes|market|sessions|throttles,activePolicy,market|sessions|throttles|stats


run.success,run.replaySafe,run.message,run.resetSummary,run.summary
True,True,scenario run complete,"{'scenarioId': 'notebook-scenario', 'resetMode': 'PURGE_AND_REPOPULATE', 'replaySafe': True, 'clearedState': 'orders|stats|outcomes|market|sessions|throttles', 'keptState': 'activePolicy', 'repopulatedState': 'market|sessions|throttles|stats'}","{'scenarioId': 'notebook-scenario', 'seed': 42, 'ticksRun': 10, 'bookChecksum': -8744230334898956430, 'outcomeCount': 20}"


order.parentOrderId,order.filledQty,order.remainingQty,order.status,order.childOrderCount,order.routeAttempts,order.timedOut
2,2500,0,3,3,1,False


fill.childOrderId,fill.parentOrderId,fill.routeAttempt,fill.instrumentId,fill.instrumentSymbol,fill.venueId,fill.venueName,fill.side,fill.sideName,fill.childQty,fill.filledQty,fill.priceTicks,fill.notionalTicks,fill.policyVersion,fill.policyHash64
2,2,1,0,INST0,0,VENUE0,1,BUY,1000,1000,10001,10001000,1,-2649222367500659138
3,2,1,0,INST0,1,VENUE1,1,BUY,1000,1000,10002,10002000,1,-2649222367500659138
4,2,1,0,INST0,2,VENUE2,1,BUY,500,500,10003,5001500,1,-2649222367500659138


summary.scenarioId,summary.seed,summary.ticksRun,summary.bookChecksum
notebook-scenario,42,10,-8744230334898956430


sequence,eventId,timestampNanos,event,componentId,eventType,correlationId,message
1,1,1,lifecycle,1,12,1,engine runtime ready
2,10000,10001,lifecycle,1,14,0,scenario reset notebook-scenario PURGE_AND_REPOPULATE
3,10002,10003,lifecycle,1,14,0,scenario reset notebook-scenario PURGE_AND_REPOPULATE
4,10004,10005,lifecycle,1,15,0,scenario run notebook-scenario checksum -8744230334898956430
5,1,21657818711055,lifecycle,1,9,1,order routed 1 childOrders=1 residual=0
6,1,21657822378544,lifecycle,1,2,1,order accepted 1
7,2,21670499582490,lifecycle,1,9,2,scenario parent order routed 2 attempt=1 childOrders=3 residual=0
